[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/01-python/10_comprehensions.ipynb)

*Python from Scratch · Notebook 10 of 12*

# Comprehensions, Iterators & Generators

Build collections in one line, use the iteration built-ins, and produce values lazily with generators.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A comprehension is a one-sentence loop. A generator is a tap that gives one value at a time, rather than a full bucket.

**What interviewers really test here:** Readability limits, lazy evaluation, memory use, and writing generators.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. List comprehensions *(Basics)*
- [ ] 2. Dict and set comprehensions *(Basics)*
- [ ] 3. Built-ins that loop for you *(Core)*
- [ ] 4. Iterables and iterators *(Core)*
- [ ] 5. Generators and yield *(Advanced)*
- [ ] 6. Generator expressions and itertools *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definitions

- A **comprehension** is a compact expression that builds a list, dict or set from an iterable, optionally filtering and transforming items: `[expr for x in iterable if condition]`.
- An **iterable** is an object that can return an iterator — it implements `__iter__` (or the sequence protocol `__getitem__`).
- An **iterator** is an object with `__next__` that returns the next item and raises `StopIteration` when exhausted; its `__iter__` returns itself.
- A **generator** is an iterator created by a function containing `yield` (a generator function) or by a generator expression `(expr for x in it)`.

### Comprehension syntax

| Kind | Syntax | Produces |
| --- | --- | --- |
| List | `[f(x) for x in xs if p(x)]` | list |
| Set | `{f(x) for x in xs}` | set |
| Dict | `{k(x): v(x) for x in xs}` | dict |
| Generator expression | `(f(x) for x in xs)` | generator (lazy) |
| Nested | `[y for row in grid for y in row]` | loops read left → right = outer → inner |
| With if/else | `[a if c else b for x in xs]` | the conditional goes before `for` |



In Python 3 the loop variable of a comprehension is local to it and doesn't leak out. Comprehensions are usually faster than the equivalent loop with `append`.

### The iterator protocol

1. `for x in obj:` calls `iter(obj)`, which calls `obj.__iter__()` and returns an iterator.
2. Each round calls `next(iterator)` → `iterator.__next__()`.
3. When `StopIteration` is raised, the loop ends quietly.



Iterators are **single-pass**: after exhaustion, further `next()` calls keep raising StopIteration. Lists, strings and dicts are iterables, not iterators — they give a fresh iterator each time, which is why you can loop over them repeatedly.

### Generators

- Calling a generator function returns a generator object without running the body.
- `next()` runs the body up to the next `yield`, returns the yielded value, and **suspends** the function, keeping its local variables and position.
- When the function returns (or falls off the end), the generator raises `StopIteration`.
- `yield from iterable` delegates to another iterable or generator.
- Generators also support `send()`, `throw()` and `close()` for coroutine-style use.



|   | List | Generator |
| --- | --- | --- |
| Evaluation | eager — all items now | lazy — one item when asked |
| Memory | O(n) | O(1) extra |
| Reusable | yes | no (single pass) |
| Indexing, len() | yes | no |
| Infinite sequences | impossible | possible |

### Functional built-ins and itertools

`map`, `filter`, `zip`, `enumerate` and `reversed` return **lazy iterators** in Python 3 — wrap them in `list()` to see all values. `sorted` returns a list. `any` and `all` short-circuit. `functools.reduce(f, xs)` folds a sequence into one value.



`itertools` highlights: `count, cycle, repeat` (infinite); `islice, takewhile, dropwhile, chain, groupby, accumulate, pairwise, zip_longest` (processing); `product, permutations, combinations` (combinatorics).

### Applications

- Transforming and filtering data in one line (cleaning a list of strings, extracting fields).
- Reading huge files line by line (a file object is an iterator).
- Data pipelines: chaining generators so each stage processes one item at a time.
- Streams and infinite sequences: IDs, sensor readings, pagination through an API.

---
# Part 2: Lessons, step by step

## Lesson 1: List comprehensions

*Level: Basics*

A **list comprehension** builds a list in one readable line. These two are the same:



```python
squares = []
for x in range(5):
    squares.append(x * x)

squares = [x * x for x in range(5)]
```



Read it as: "*x \* x*, **for** each x **in** range(5)". Add a filter at the end:



```python
evens = [x for x in nums if x % 2 == 0]
```



And choose between two values with a conditional expression at the front:



```python
labels = ["even" if x % 2 == 0 else "odd" for x in nums]
```

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def list_comp_demo():
    nums = [3, 8, 5, 12]
    squares = [x * x for x in nums]
    evens = [x for x in nums if x % 2 == 0]
    labels = ["big" if x > 6 else "small" for x in nums]
    print(squares, evens, labels)


list_comp_demo()

[9, 64, 25, 144] [8, 12] ['small', 'big', 'small', 'big']


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q5](#q5), [Q7](#q7). Try one now, or carry on and do them in Part 3.

## Lesson 2: Dict and set comprehensions

*Level: Basics*

```python
lengths = {w: len(w) for w in words}          # dict: key: value
unique_lengths = {len(w) for w in words}       # set: duplicates vanish
inverted = {v: k for k, v in d.items()}
```



Nested comprehensions read in the same order as nested loops:



```python
grid = [[r * c for c in range(3)] for r in range(3)]   # list of rows
flat = [x for row in grid for x in row]                # flatten
```



**💡 Tip:** If a comprehension no longer fits comfortably on one line, a normal loop is clearer.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def other_comp_demo():
    words = ["hi", "there", "you", "all"]
    lengths = {w: len(w) for w in words}
    unique_lengths = {len(w) for w in words}
    grid = [[r * c for c in range(1, 3)] for r in range(1, 3)]
    flat = [x for row in grid for x in row]
    print(lengths, unique_lengths, grid, flat)


other_comp_demo()

{'hi': 2, 'there': 5, 'you': 3, 'all': 3} {2, 3, 5} [[1, 2], [2, 4]] [1, 2, 2, 4]


**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q4](#q4), [Q13](#q13). Try one now, or carry on and do them in Part 3.

## Lesson 3: Built-ins that loop for you

*Level: Core*

| Built-in | Does | Example |
| --- | --- | --- |
| `enumerate(xs, start=0)` | (index, item) pairs | `for i, x in enumerate(xs)` |
| `zip(a, b)` | items side by side | `for x, y in zip(a, b)` |
| `map(f, xs)` | apply f to each item | `map(int, ["1", "2"])` |
| `filter(f, xs)` | keep items where f is true | `filter(str.isdigit, parts)` |
| `sorted(xs, key=, reverse=)` | new sorted list | `sorted(people, key=lambda p: (p[1], p[0]))` |
| `any(xs)` / `all(xs)` | is any / every item truthy? | `all(x > 0 for x in nums)` |
| `sum`, `min`, `max` | aggregate | `max(words, key=len)` |
| `reversed(xs)` | iterate backwards | `for x in reversed(xs)` |

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def builtins_demo():
    names = ["ana", "ben", "cy"]
    ages = [25, 31, 19]
    for i, (name, age) in enumerate(zip(names, ages), start=1):
        print(i, name, age)
    adults = list(filter(lambda a: a >= 21, ages))
    upper = list(map(str.upper, names))
    print(adults, upper, any(a < 20 for a in ages), all(a > 18 for a in ages))


builtins_demo()

1 ana 25
2 ben 31
3 cy 19
[25, 31] ['ANA', 'BEN', 'CY'] True True


**✏️ Your turn:** you now have what you need for [Q6](#q6), [Q14](#q14), [Q8](#q8), [Q15](#q15), [Q12](#q12). Try one now, or carry on and do them in Part 3.

## Lesson 4: Iterables and iterators

*Level: Core*

An **iterable** is anything you can loop over (list, str, dict, range, file). An **iterator** is the object that actually hands out the items one at a time and remembers where it is.



```python
nums = [10, 20]
it = iter(nums)      # get an iterator
next(it)             # 10
next(it)             # 20
next(it)             # StopIteration — no more items
```



A `for` loop does exactly this behind the scenes. An iterator can be used **only once**; to loop again, ask the iterable for a new one.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def iterator_demo():
    nums = [10, 20, 30]
    it = iter(nums)
    first = next(it)
    second = next(it)
    rest = list(it)
    again = list(it)
    print(first, second, rest, again)


iterator_demo()

10 20 [30] []


**✏️ Your turn:** you now have what you need for [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 5: Generators and yield

*Level: Advanced*

A **generator function** contains `yield`. Calling it doesn't run the body — it returns a generator. Each `next()` runs the body until the next `yield`, hands out that value, and **pauses** there with all its variables kept.



```python
def countdown(n):
    while n > 0:
        yield n        # pause here, give n to the caller
        n -= 1         # resume from here on the next request

for x in countdown(3):
    print(x)           # 3, 2, 1
```



In the animation, watch the generator's frame appear on the call stack each time a value is requested, and keep `n` between requests.



Generators are **lazy**: they produce values only when asked, so they can represent huge or even infinite sequences using almost no memory.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def generator_demo():
    def countdown(n):
        while n > 0:
            yield n
            n -= 1

    gen = countdown(3)
    print(next(gen))
    for x in gen:
        print("loop got", x)


generator_demo()

3
loop got 2
loop got 1


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. countdown() returns 3
2. countdown() returns 2
3. countdown() returns 1
4. countdown() returns None
5. generator\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q9](#q9), [Q19](#q19), [Q18](#q18), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 6: Generator expressions and itertools

*Level: Advanced*

A **generator expression** looks like a list comprehension with round brackets. It yields items lazily instead of building a list:



```python
total = sum(x * x for x in range(1_000_000))   # no million-item list in memory
```



The `itertools` module has ready-made iterator tools:



| Tool | Does |
| --- | --- |
| `count(start, step)` | endless counter |
| `islice(it, n)` | first n items of any iterator |
| `chain(a, b, …)` | one iterable after another |
| `groupby(it, key)` | runs of equal items |
| `product`, `permutations`, `combinations` | combinatorics |
| `accumulate(it)` | running totals |
| `pairwise(it)` | neighbouring pairs (3.10+) |

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [7]:
def itertools_demo():
    from itertools import count, islice, accumulate
    evens = (n for n in count(0, 2))
    first_five = list(islice(evens, 5))
    totals = list(accumulate([1, 2, 3, 4]))
    big_sum = sum(x * x for x in range(1, 101))
    print(first_five, totals, big_sum)


itertools_demo()

[0, 2, 4, 6, 8] [1, 3, 6, 10] 338350


**✏️ Your turn:** you now have what you need for [Q16](#q16), [Q10](#q10), [Q11](#q11). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Squares

🟢 Warm-up · *List comprehension*

Return the squares of the numbers in the list.

**Examples**

- `nums = [1, 2, 3]` → `[1, 4, 9]`
- `nums = []` → `[]`

In [ ]:
def squares(nums):
    """
    Return the squares of the numbers in the list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(squares, [
    (([1, 2, 3],), [1, 4, 9]),
    (([],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and append.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`[x * x for x in nums]`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop and append.

```python
def squares(nums):
    result = []
    for x in nums:
        result.append(x * x)
    return result
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `[x * x for x in nums]`.

```python
def squares(nums):
    return [x * x for x in nums]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and append. That costs O(n) time and O(n) space."

"I can do better: `[x * x for x in nums]`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/squares)

<a id="q2"></a>
### Q2. Keep the Even Numbers

🟢 Warm-up · *Filtering*

Return only the even numbers, in order.

**Examples**

- `nums = [1, 2, 3, 4, 6]` → `[2, 4, 6]`
- `nums = [1, 3]` → `[]`

In [ ]:
def evens(nums):
    """
    Return only the even numbers, in order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(evens, [
    (([1, 2, 3, 4, 6],), [2, 4, 6]),
    (([1, 3],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with an if and append.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension with an `if` clause.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop with an if and append.

```python
def evens(nums):
    out = []
    for x in nums:
        if x % 2 == 0:
            out.append(x)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A comprehension with an `if` clause.

```python
def evens(nums):
    return [x for x in nums if x % 2 == 0]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with an if and append. That costs O(n) time and O(n) space."

"I can do better: A comprehension with an `if` clause. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/keep-the-even-numbers)

<a id="q3"></a>
### Q3. Word Lengths

🟢 Warm-up · *Dict comprehension*

Return a dict mapping each word to its length.

**Examples**

- `words = ['hi', 'there']` → `{'hi': 2, 'there': 5}`
- `words = []` → `{}`

In [ ]:
def word_lengths(words):
    """
    Return a dict mapping each word to its length.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(word_lengths, [
    ((['hi', 'there'],), {'hi': 2, 'there': 5}),
    (([],), {}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and assign keys.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`{w: len(w) for w in words}`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop and assign keys.

```python
def word_lengths(words):
    d = {}
    for w in words:
        d[w] = len(w)
    return d
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `{w: len(w) for w in words}`.

```python
def word_lengths(words):
    return {w: len(w) for w in words}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and assign keys. That costs O(n) time and O(n) space."

"I can do better: `{w: len(w) for w in words}`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/word-lengths)

<a id="q4"></a>
### Q4. Distinct Word Lengths

🟢 Warm-up · *Set comprehension*

Return the sorted list of different word lengths in a sentence.

**Examples**

- `sentence = 'the cat sat on a mat'` → `[1, 2, 3]`
- `sentence = ''` → `[]`

In [ ]:
def distinct_lengths(sentence):
    """
    Return the sorted list of different word lengths in a sentence.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(distinct_lengths, [
    (('the cat sat on a mat',), [1, 2, 3]),
    (('',), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Collect lengths into a list, skipping ones already present, then sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set comprehension removes duplicates automatically.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Collect lengths into a list, skipping ones already present, then sort.

```python
def distinct_lengths(sentence):
    lengths = []
    for w in sentence.split():
        if len(w) not in lengths:
            lengths.append(len(w))
    lengths.sort()
    return lengths
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A set comprehension removes duplicates automatically.

```python
def distinct_lengths(sentence):
    return sorted({len(w) for w in sentence.split()})
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Collect lengths into a list, skipping ones already present, then sort. That costs O(n²) time and O(n) space."

"I can do better: A set comprehension removes duplicates automatically. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/distinct-word-lengths)

<a id="q5"></a>
### Q5. Label Even and Odd

🟢 Warm-up · *if/else inside a comprehension*

Return 'even' or 'odd' for each number.

**Examples**

- `nums = [1, 2, 3]` → `['odd', 'even', 'odd']`
- `nums = []` → `[]`

In [ ]:
def labels(nums):
    """
    Return 'even' or 'odd' for each number.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(labels, [
    (([1, 2, 3],), ['odd', 'even', 'odd']),
    (([],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with if/else and append.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A conditional expression before the `for`: `['even' if x % 2 == 0 else 'odd' for x in nums]`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop with if/else and append.

```python
def labels(nums):
    out = []
    for x in nums:
        if x % 2 == 0:
            out.append("even")
        else:
            out.append("odd")
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A conditional expression before the `for`: `['even' if x % 2 == 0 else 'odd' for x in nums]`.

```python
def labels(nums):
    return ["even" if x % 2 == 0 else "odd" for x in nums]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with if/else and append. That costs O(n) time and O(n) space."

"I can do better: A conditional expression before the `for`: `['even' if x % 2 == 0 else 'odd' for x in nums]`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/label-even-and-odd)

<a id="q6"></a>
### Q6. Pairwise Sums

🟢 Warm-up · *zip*

Return [a[0] + b[0], a[1] + b[1], …] for two lists of equal length.

**Examples**

- `a = [1, 2, 3], b = [10, 20, 30]` → `[11, 22, 33]`
- `a = [], b = []` → `[]`

In [ ]:
def pairwise_sums(a, b):
    """
    Return [a[0] + b[0], a[1] + b[1], …] for two lists of equal length.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pairwise_sums, [
    (([1, 2, 3], [10, 20, 30]), [11, 22, 33]),
    (([], []), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over indices.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`[x + y for x, y in zip(a, b)]`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over indices.

```python
def pairwise_sums(a, b):
    out = []
    for i in range(len(a)):
        out.append(a[i] + b[i])
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `[x + y for x, y in zip(a, b)]`.

```python
def pairwise_sums(a, b):
    return [x + y for x, y in zip(a, b)]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over indices. That costs O(n) time and O(n) space."

"I can do better: `[x + y for x, y in zip(a, b)]`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/pairwise-sums)

<a id="q7"></a>
### Q7. All Positions of a Value

🟢 Warm-up · *enumerate*

Return every index where `value` appears.

**Examples**

- `items = [1, 3, 1, 2, 1], value = 1` → `[0, 2, 4]`
- `items = ['a'], value = 'b'` → `[]`

In [ ]:
def positions(items, value):
    """
    Return every index where value appears.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(positions, [
    (([1, 3, 1, 2, 1], 1), [0, 2, 4]),
    ((['a'], 'b'), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over range(len()) and compare.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`[i for i, x in enumerate(items) if x == value]`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over range(len()) and compare.

```python
def positions(items, value):
    out = []
    for i in range(len(items)):
        if items[i] == value:
            out.append(i)
    return out
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `[i for i, x in enumerate(items) if x == value]`.

```python
def positions(items, value):
    return [i for i, x in enumerate(items) if x == value]
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over range(len()) and compare. That costs O(n) time and O(k) space."

"I can do better: `[i for i, x in enumerate(items) if x == value]`. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/all-positions-of-a-value)

<a id="q8"></a>
### Q8. All Positive?

🟢 Warm-up · *all() and any()*

Return True if every number is positive.

**Examples**

- `nums = [1, 5, 9]` → `True`
- `nums = [3, -1]` → `False`
- `nums = []` → `True`

In [ ]:
def all_positive(nums):
    """
    Return True if every number is positive.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(all_positive, [
    (([1, 5, 9],), True),
    (([3, -1],), False),
    (([],), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and return False at the first non-positive number.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`all(x > 0 for x in nums)` — it also stops at the first False.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop and return False at the first non-positive number.

```python
def all_positive(nums):
    for x in nums:
        if x <= 0:
            return False
    return True
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `all(x > 0 for x in nums)` — it also stops at the first False.

```python
def all_positive(nums):
    return all(x > 0 for x in nums)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and return False at the first non-positive number. That costs O(n) time and O(1) space."

"I can do better: `all(x > 0 for x in nums)` — it also stops at the first False. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/all-positive)

<a id="q9"></a>
### Q9. First n Squares (Generator)

🟢 Warm-up · *yield*

Write a generator that yields 1², 2², …, n². (Shown as a list.)

**Examples**

- `n = 4` → `[1, 4, 9, 16]`
- `n = 0` → `[]`

In [ ]:
def squares_upto(n):
    """
    Write a generator that yields 1², 2², …, n². (Shown as a list.)
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(squares_upto, [
    ((4,), [1, 4, 9, 16]),
    ((0,), []),
], listify=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build and return the whole list at once.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`yield` each square — values are produced one at a time, on demand.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Build and return the whole list at once.

```python
def squares_upto(n):
    result = []
    for i in range(1, n + 1):
        result.append(i * i)
    return result
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `yield` each square — values are produced one at a time, on demand.

```python
def squares_upto(n):
    for i in range(1, n + 1):
        yield i * i
```

**Cost:** O(n) time, O(1) per value space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build and return the whole list at once. That costs O(n) time and O(n) space."

"I can do better: `yield` each square — values are produced one at a time, on demand. That's O(n) time and O(1) per value space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 4`

1. squares\_upto() returns 1
2. squares\_upto() returns 4
3. squares\_upto() returns 9
4. squares\_upto() returns 16
5. squares\_upto() returns None

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/first-n-squares-generator)

<a id="q10"></a>
### Q10. Sum of Squares Without a List

🟢 Warm-up · *Generator expression*

Return 1² + 2² + … + n².

**Examples**

- `n = 3` → `14`
- `n = 10` → `385`
- `n = 0` → `0`

In [ ]:
def sum_squares(n):
    """
    Return 1² + 2² + … + n².
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sum_squares, [
    ((3,), 14),
    ((10,), 385),
    ((0,), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build the full list of squares, then sum it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A generator expression inside `sum` produces each square on demand — no list in memory.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Build the full list of squares, then sum it.

```python
def sum_squares(n):
    squares = [i * i for i in range(1, n + 1)]
    return sum(squares)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A generator expression inside `sum` produces each square on demand — no list in memory.

```python
def sum_squares(n):
    return sum(i * i for i in range(1, n + 1))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build the full list of squares, then sum it. That costs O(n) time and O(n) space."

"I can do better: A generator expression inside `sum` produces each square on demand — no list in memory. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/sum-of-squares-without-a-list)

<a id="q11"></a>
### Q11. Chain Several Lists

🟢 Warm-up · *itertools.chain*

Return one list with the items of all the given lists in order.

**Examples**

- `lists = [1, 2]` → `[1, 2, 3, 4, 5]`
- `lists = []` → `[]`

In [ ]:
def chain_all(*lists):
    """
    Return one list with the items of all the given lists in order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(chain_all, [
    (([1, 2], [3], [4, 5]), [1, 2, 3, 4, 5]),
    (([],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over each list and extend a result.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`itertools.chain(*lists)` walks them one after another without copying.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over each list and extend a result.

```python
def chain_all(*lists):
    out = []
    for lst in lists:
        for x in lst:
            out.append(x)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `itertools.chain(*lists)` walks them one after another without copying.

```python
def chain_all(*lists):
    from itertools import chain
    return list(chain(*lists))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over each list and extend a result. That costs O(n) time and O(n) space."

"I can do better: `itertools.chain(*lists)` walks them one after another without copying. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/chain-several-lists)

<a id="q12"></a>
### Q12. Adjacent Differences

🟢 Warm-up · *zip with a shifted copy*

Return [b − a for each neighbouring pair (a, b)].

**Examples**

- `nums = [1, 4, 9, 16]` → `[3, 5, 7]`
- `nums = [5]` → `[]`
- `nums = []` → `[]`

In [ ]:
def differences(nums):
    """
    Return [b − a for each neighbouring pair (a, b)].
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(differences, [
    (([1, 4, 9, 16],), [3, 5, 7]),
    (([5],), []),
    (([],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over indices 0 … n − 2.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Pair each item with the next using `zip(nums, nums[1:])` (or `itertools.pairwise`).

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over indices 0 … n − 2.

```python
def differences(nums):
    out = []
    for i in range(len(nums) - 1):
        out.append(nums[i + 1] - nums[i])
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Pair each item with the next using `zip(nums, nums[1:])` (or `itertools.pairwise`).

```python
def differences(nums):
    return [b - a for a, b in zip(nums, nums[1:])]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over indices 0 … n − 2. That costs O(n) time and O(n) space."

"I can do better: Pair each item with the next using `zip(nums, nums[1:])` (or `itertools.pairwise`). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/adjacent-differences)

<a id="q13"></a>
### Q13. Multiplication Grid

🟡 Interview standard · *Nested comprehension*

Return an n × n grid where cell (r, c) is (r + 1) × (c + 1).

**Examples**

- `n = 3` → `[[1, 2, 3], [2, 4, 6], [3, 6, 9]]`
- `n = 1` → `[[1]]`

In [ ]:
def grid(n):
    """
    Return an n × n grid where cell (r, c) is (r + 1) × (c + 1).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(grid, [
    ((3,), [[1, 2, 3], [2, 4, 6], [3, 6, 9]]),
    ((1,), [[1]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two nested loops building rows.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension inside a comprehension: the outer one builds rows.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Two nested loops building rows.

```python
def grid(n):
    rows = []
    for r in range(1, n + 1):
        row = []
        for c in range(1, n + 1):
            row.append(r * c)
        rows.append(row)
    return rows
```

**Cost:** O(n²) time, O(n²) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A comprehension inside a comprehension: the outer one builds rows.

```python
def grid(n):
    return [[r * c for c in range(1, n + 1)] for r in range(1, n + 1)]
```

**Cost:** O(n²) time, O(n²) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two nested loops building rows. That costs O(n²) time and O(n²) space."

"I can do better: A comprehension inside a comprehension: the outer one builds rows. That's O(n²) time and O(n²) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/multiplication-grid)

<a id="q14"></a>
### Q14. Parse and Keep Positives

🟡 Interview standard · *map and filter*

Given number strings like ['3', '-1', '7'], return the positive values as ints.

**Examples**

- `texts = ['3', '-1', '7', '0']` → `[3, 7]`
- `texts = []` → `[]`

In [ ]:
def positives(texts):
    """
    Given number strings like ['3', '-1', '7'], return the positive values as
    ints.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(positives, [
    ((['3', '-1', '7', '0'],), [3, 7]),
    (([],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop, convert, test, append.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`filter(lambda x: x > 0, map(int, texts))` — or a comprehension.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop, convert, test, append.

```python
def positives(texts):
    out = []
    for t in texts:
        value = int(t)
        if value > 0:
            out.append(value)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `filter(lambda x: x > 0, map(int, texts))` — or a comprehension.

```python
def positives(texts):
    return list(filter(lambda x: x > 0, map(int, texts)))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop, convert, test, append. That costs O(n) time and O(n) space."

"I can do better: `filter(lambda x: x > 0, map(int, texts))` — or a comprehension. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/parse-and-keep-positives)

<a id="q15"></a>
### Q15. Sort People by Age, Then Name

🟡 Interview standard · *Tuple keys*

Sort (name, age) pairs by age ascending, and by name for equal ages.

**Examples**

- `people = [('cy', 30), ('ana', 25), ('ben', 25)]` → `[('ana', 25), ('ben', 25), ('cy', 30)]`
- `people = []` → `[]`

In [ ]:
def sort_people(people):
    """
    Sort (name, age) pairs by age ascending, and by name for equal ages.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sort_people, [
    (([('cy', 30), ('ana', 25), ('ben', 25)],), [('ana', 25), ('ben', 25), ('cy', 30)]),
    (([],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort by name first, then by age — Python's sort is stable, so names stay ordered within each age.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One sort with a tuple key `(age, name)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Sort by name first, then by age — Python's sort is stable, so names stay ordered within each age.

```python
def sort_people(people):
    by_name = sorted(people, key=lambda p: p[0])
    return sorted(by_name, key=lambda p: p[1])
```

**Cost:** O(n log n) × 2 time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** One sort with a tuple key `(age, name)`.

```python
def sort_people(people):
    return sorted(people, key=lambda p: (p[1], p[0]))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort by name first, then by age — Python's sort is stable, so names stay ordered within each age. That costs O(n log n) × 2 time and O(n) space."

"I can do better: One sort with a tuple key `(age, name)`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/sort-people-by-age-then-name)

<a id="q16"></a>
### Q16. Take From an Endless Stream

🟡 Interview standard · *Infinite generator · islice*

Return the first `k` even numbers starting from 0, using an endless counter in the optimal answer.

**Examples**

- `k = 5` → `[0, 2, 4, 6, 8]`
- `k = 0` → `[]`

In [ ]:
def first_evens(k):
    """
    Return the first k even numbers starting from 0, using an endless counter in
    the optimal answer.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(first_evens, [
    ((5,), [0, 2, 4, 6, 8]),
    ((0,), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A while loop with a counter and a list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

An infinite generator plus `itertools.islice` to take just k values.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A while loop with a counter and a list.

```python
def first_evens(k):
    out, n = [], 0
    while len(out) < k:
        out.append(n)
        n += 2
    return out
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** An infinite generator plus `itertools.islice` to take just k values.

```python
def first_evens(k):
    from itertools import count, islice

    def evens():
        for n in count(0, 2):
            yield n

    return list(islice(evens(), k))
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A while loop with a counter and a list. That costs O(k) time and O(k) space."

"I can do better: An infinite generator plus `itertools.islice` to take just k values. That's O(k) time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `k = 4`

1. evens() returns 0
2. evens() returns 2
3. evens() returns 4
4. evens() returns 6
5. GeneratorExit raised: 
6. evens() returns None
7. first\_evens() returns [0, 2, 4, 6]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/take-from-an-endless-stream)

<a id="q17"></a>
### Q17. Running Average

🟡 Interview standard · *Generator with state*

Return the running averages of a list (rounded to 2 decimals) using a generator in the optimal answer.

**Examples**

- `nums = [2, 4, 6]` → `[2.0, 3.0, 4.0]`
- `nums = [5]` → `[5.0]`
- `nums = []` → `[]`

In [ ]:
def running_avg(nums):
    """
    Return the running averages of a list (rounded to 2 decimals) using a
    generator in the optimal answer.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(running_avg, [
    (([2, 4, 6],), [2.0, 3.0, 4.0]),
    (([5],), [5.0]),
    (([],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each position, average the slice up to it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A generator keeps a running total and count between yields.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** For each position, average the slice up to it.

```python
def running_avg(nums):
    out = []
    for i in range(len(nums)):
        part = nums[:i + 1]
        out.append(round(sum(part) / len(part), 2))
    return out
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A generator keeps a running total and count between yields.

```python
def running_avg(nums):
    def averages():
        total = 0
        for count, x in enumerate(nums, start=1):
            total += x
            yield round(total / count, 2)

    return list(averages())
```

**Cost:** O(n) time, O(1) state space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each position, average the slice up to it. That costs O(n²) time and O(n) space."

"I can do better: A generator keeps a running total and count between yields. That's O(n) time and O(1) state space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [2, 4, 6]`

1. averages() returns 2.0
2. averages() returns 3.0
3. averages() returns 4.0
4. averages() returns None
5. running\_avg() returns [2.0, 3.0, 4.0]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/running-average)

<a id="q18"></a>
### Q18. Batches (Generator)

🟡 Interview standard · *yield slices*

Yield the items in batches of size k (last batch may be shorter). Shown as a list.

**Examples**

- `items = [1, 2, 3, 4, 5], k = 2` → `[[1, 2], [3, 4], [5]]`
- `items = [], k = 3` → `[]`

In [ ]:
def batches(items, k):
    """
    Yield the items in batches of size k (last batch may be shorter). Shown as a
    list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(batches, [
    (([1, 2, 3, 4, 5], 2), [[1, 2], [3, 4], [5]]),
    (([], 3), []),
], listify=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build all the batches in a list first.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Yield one slice at a time — memory stays small even for huge inputs.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Build all the batches in a list first.

```python
def batches(items, k):
    result = []
    for i in range(0, len(items), k):
        result.append(items[i:i + k])
    return result
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Yield one slice at a time — memory stays small even for huge inputs.

```python
def batches(items, k):
    for i in range(0, len(items), k):
        yield items[i:i + k]
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build all the batches in a list first. That costs O(n) time and O(n) space."

"I can do better: Yield one slice at a time — memory stays small even for huge inputs. That's O(n) time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `items = [1, 2, 3, 4, 5], k = 2`

1. batches() returns [1, 2]
2. batches() returns [3, 4]
3. batches() returns [5]
4. batches() returns None

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/batches-generator)

<a id="q19"></a>
### Q19. Fibonacci Up To a Limit

🟡 Interview standard · *Generator*

Yield Fibonacci numbers 0, 1, 1, 2, … that are ≤ limit. Shown as a list.

**Examples**

- `limit = 10` → `[0, 1, 1, 2, 3, 5, 8]`
- `limit = 0` → `[0]`

In [ ]:
def fib_upto(limit):
    """
    Yield Fibonacci numbers 0, 1, 1, 2, … that are ≤ limit. Shown as a list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fib_upto, [
    ((10,), [0, 1, 1, 2, 3, 5, 8]),
    ((0,), [0]),
], listify=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build a list with a while loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A generator with two running variables.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Build a list with a while loop.

```python
def fib_upto(limit):
    seq = [0, 1]
    while seq[-1] + seq[-2] <= limit:
        seq.append(seq[-1] + seq[-2])
    return [x for x in seq if x <= limit]
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A generator with two running variables.

```python
def fib_upto(limit):
    a, b = 0, 1
    while a <= limit:
        yield a
        a, b = b, a + b
```

**Cost:** O(k) time, O(1) state space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build a list with a while loop. That costs O(k) time and O(k) space."

"I can do better: A generator with two running variables. That's O(k) time and O(1) state space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `limit = 10`

1. fib\_upto() returns 0
2. fib\_upto() returns 1
3. fib\_upto() returns 1
4. fib\_upto() returns 2
5. fib\_upto() returns 3
6. fib\_upto() returns 5
7. fib\_upto() returns 8
8. fib\_upto() returns None

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/fibonacci-up-to-a-limit)

<a id="q20"></a>
### Q20. Countdown Iterator

🟡 Interview standard · *Iterator protocol vs generator*

Produce n, n − 1, …, 1 as an iterator. (Shown as a list.)

**Examples**

- `n = 3` → `[3, 2, 1]`
- `n = 0` → `[]`

In [ ]:
def countdown(n):
    """
    Produce n, n − 1, …, 1 as an iterator. (Shown as a list.)
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(countdown, [
    ((3,), [3, 2, 1]),
    ((0,), []),
], listify=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A class implementing `__iter__` and `__next__`, raising StopIteration at the end.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A generator function does the same in three lines — Python writes \_\_next\_\_ for you.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A class implementing `__iter__` and `__next__`, raising StopIteration at the end.

```python
def countdown(n):
    class Countdown:
        def __init__(self, start):
            self.current = start

        def __iter__(self):
            return self

        def __next__(self):
            if self.current <= 0:
                raise StopIteration
            value = self.current
            self.current -= 1
            return value

    return Countdown(n)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A generator function does the same in three lines — Python writes \_\_next\_\_ for you.

```python
def countdown(n):
    while n > 0:
        yield n
        n -= 1
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A class implementing `__iter__` and `__next__`, raising StopIteration at the end. That costs O(n) time and O(1) space."

"I can do better: A generator function does the same in three lines — Python writes \_\_next\_\_ for you. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 3`

1. countdown() returns 3
2. countdown() returns 2
3. countdown() returns 1
4. countdown() returns None

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/comprehensions/countdown-iterator)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. List comprehension vs generator expression?</b></summary>

Square brackets build the whole list immediately (uses memory, reusable, indexable). Round brackets create a lazy generator that produces items on demand (tiny memory, single use).

</details>

<details>
<summary><b>2. What is the difference between an iterable and an iterator?</b></summary>

An iterable can produce an iterator (`iter(x)` works) and can usually be looped over many times. An iterator produces the items with `next()`, remembers its position and is used up after one pass.

</details>

<details>
<summary><b>3. What does yield do?</b></summary>

It hands a value to the caller and pauses the function, keeping its state. The next request resumes right after the yield.

</details>

<details>
<summary><b>4. return vs yield?</b></summary>

`return` ends the function and sends back one value. `yield` produces a value but keeps the function alive so it can produce more later.

</details>

<details>
<summary><b>5. When should you use a generator?</b></summary>

When the sequence is large, infinite, or expensive to build, and you only need to go through it once — e.g. processing a big log file line by line.

</details>

<details>
<summary><b>6. Where does the if go in a comprehension?</b></summary>

A filter goes at the end: `[x for x in xs if x > 0]`. A choice between two values goes at the front: `[x if x > 0 else 0 for x in xs]`.

</details>

<details>
<summary><b>7. Are comprehensions faster than loops?</b></summary>

Usually a little, because the loop runs in optimised bytecode without repeated `append` attribute lookups. Readability matters more: use a normal loop when the logic is complex.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: List comprehensions
- [ ] I can explain: Dict and set comprehensions
- [ ] I can explain: Built-ins that loop for you
- [ ] I can explain: Iterables and iterators
- [ ] I can explain: Generators and yield
- [ ] I can explain: Generator expressions and itertools

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Take From an Endless Stream
- [ ] [Q17](#q17) Running Average
- [ ] [Q18](#q18) Batches (Generator)
- [ ] [Q19](#q19) Fibonacci Up To a Limit
- [ ] [Q20](#q20) Countdown Iterator

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.